[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/electromagnetism/lie_advection/lie_advection.ipynb)

# Discrete Lie Advection

This example computes Lie advection, the transport of a magnetic field by a moving conductor: in a perfect conductor the field is frozen into the material and carried along with it, here across a fixed triangle mesh. A moving edge sweeps a sheet through spacetime, and the field on that sheet changes the edge's circulation reading; around a closed boundary, the contributions at the edges' endpoints cancel. A prescribed vortex winds positive and negative magnetic flux into fine layers, while resistivity smooths them.

The mesh lives in the spacetime of its plane, `x+y+t-`. The magnetic field is a bivector, an extrinsic plane of space, `mv.xy`; an electric field is a plane through time. An edge moving at `velocity` sweeps the world sheet `(time + velocity) & edge`, an intrinsic patch, and the field read on that sheet, their regressive product with no metric, is the rate of the edge's reading. In an ideal conductor, by Ohm's law, the field vanishes on the world sheets of edges that move with it; so a fixed edge's reading, read on `time & edge`, changes by the magnetic flux through the patch of space between its world sheet and the comoving one. Whitney reconstruction and the curl's adjugate turn these local readings into an evolution map on the mesh.

In vector-calculus notation, the magnetic field of a conductor moving at velocity $\mathbf v$ with resistivity $\eta$ obeys the induction equation $\partial_t\mathbf B = \nabla\times(\mathbf v\times\mathbf B) - \nabla\times(\eta\,\nabla\times\mathbf B)$; in differential-form notation its ideal part is the frozen-in condition $\partial_t B + \mathcal L_v B = 0$, with the Lie derivative $\mathcal L_v$ along the flow.


In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from itertools import islice

import numpy as np

from numga import stack
from numga.algebras import STA2D as ga
from numga.sparse import SparseExtensor, spdiag
from examples import instantiate
from examples.electromagnetism.lie_advection import render, scenarios
from examples.util.animation import save_figure, show_animation

mesh_module = instantiate("examples.util.mesh", ga)
as_scalar, mv = mesh_module.as_scalar, mesh_module.mv
Scalar = ga.gatype.scalar()                                                  # Scalar+
Bivector = ga.gatype.bivector()                                              # Bivector+
# The magnetic field, a plane of space.
Magnetic = Bivector.derive.within("x y")                                     # Bivector+
Point = ga.gatype.antivector()                                               # Antivector-
Amount = ga.gatype.antiscalar()                                              # Antiscalar-
Evolution = ga.gatype((Scalar, Scalar))

# The direction in time of an observer at rest with the mesh, which pairs to one with `mv.t`.
time = -mv.t.dual()                                                          # [] Point-

np.set_printoptions(precision=4, suppress=True)

## Swept flux and endpoint readings

The reading along a moving edge changes both at its endpoints, where the potential is read along their world lines, and through the world sheet the edge sweeps. Around a triangle, each endpoint belongs to two oppositely oriented edges, so its two contributions cancel. The closed-loop change is the flux through the sheets the boundary sweeps.


In [ ]:
horizontal, vertical = mv.x.dual(), mv.y.dual()                              # [] Point-, Point-
ribbon_step = 0.55
corner_forms = mv.vector([[0.45, -0.65, 0.0], [1.35, 0.25, 0.0], [-0.55, 1.15, 0.0]])  # [corners] Plane+
corner_velocities = stack((0.48 * horizontal + 0.26 * vertical,
                           0.62 * horizontal + 0.34 * vertical,
                           0.38 * horizontal + 0.44 * vertical))             # [corners] Point-

In [ ]:
corners = scenarios.single_triangle()                                         # [corners] Point-
sides = corners[[2, 0, 1]] - corners[[1, 2, 0]]                              # [corners] Point-
patch = (sides[0] & sides[1]) / 2                                            # [] Patch-
# Each hat is constant in time: its gradient is the plane through the opposite side and time,
# over twice the prism the triangle sweeps in a unit of time.
gradients = (time & sides) / (2 * (time & patch))                            # [corners] Plane+

# Wedging each corner hat gradient with its corner 1-form sums to the constant 2-form curl on the face.
face_bivector = (gradients ^ corner_forms).sum(axis=0)                       # [] Bivector+
# Each corner advances through spacetime along its world line, `time + velocity` per unit time.
corner_motions = time + corner_velocities                                    # [corners] Point-
interior_readings = corner_forms & corner_motions                            # [corners] Scalar+
endpoint_differences = interior_readings[[2, 0, 1]] - interior_readings[[1, 2, 0]]  # [corners] Scalar+
edge_motions = (corner_motions[[1, 2, 0]] + corner_motions[[2, 0, 1]]) / 2   # [corners] Point-
world_sheets = edge_motions & sides                                          # [corners] Patch-

# Pairing the 2-form curl with each edge's world sheet gives the magnetic flux crossing the moving edge.
swept_fluxes = face_bivector & world_sheets                                  # [corners] Scalar+

save_figure(
    render.draw_cartan_ribbon(corners, corner_forms, corner_velocities,
                              ribbon_step, endpoint_differences, swept_fluxes),
    "lie_advection_cartan",
);

Around the closed triangle only the swept flux remains: the endpoint contributions, in blue, cancel in the loop sum, so the total, dark, equals the swept flux, orange. Each orange arrow carries a corner to the dashed outline, and each edge sweeps a shaded ribbon, the trace its world sheet leaves in space.

In differential-form notation, this decomposition reads as Cartan's formula $\mathcal{L}_v A = d(\iota_v A) + \iota_v(dA)$; its integral along a moving edge is $\frac{d}{dt}\int_{e(t)} A = \int_{\partial e} \iota_v A + \int_e \iota_v(dA)$.


In [ ]:
print("Endpoint differences:", endpoint_differences.to_array())
print("World-sheet fluxes:", swept_fluxes.to_array())


## The disk and its vortex

A prescribed incompressible vortex carries signed magnetic flux around the disk. The vortex follows the level curves of a streamfunction whose value and slope vanish at the rim, so the fluid at the rim stands still. The magnetic field starts as two lobes of opposite flux.


In [ ]:
n_disk_divisions = 16
disk_radius = 1.0
swirl_rate = 5.2
ellipticity = 0.2
lobe_centres = stack((-0.36 * horizontal - 0.06 * vertical,
                      0.36 * horizontal + 0.06 * vertical))                  # [lobes] Point-
lobe_strengths = np.array([1.0, -0.65])                                      # [lobes]
lobe_radius = 0.28

In [ ]:
mesh, vertices = scenarios.disk(disk_radius, n_disk_divisions)               # Mesh, Point-[vertices]
d1 = mesh.d[1] * Scalar                                                      # (Scalar+ <- Scalar+)[faces <- edges]
edges = mesh.d[0] * vertices                                                 # Point-[edges]
edge_midpoints = mesh.incidence[1, 0].mean(axis=0) * vertices                # Point-[edges]
face_sides = mesh.boundary[2, 1] * edges                                     # [corners] Point-[faces]
patches = (face_sides[0] & face_sides[1]) / 2                                # Patch-[faces]
# The prism each face sweeps in a unit of time; its size is the face's area.
prisms = time & patches                                                      # Scalar-[faces]
areas = prisms.norm()                                                        # Scalar+[faces]
flux_density = spdiag(1 / areas) * d1                                        # (Scalar+ <- Scalar+)[faces <- edges]
normals = patches / areas                                                    # Patch-[faces]

# Each corner hat's gradient is constant across its face; assembling by corner maps
# vertex streamfunctions to piecewise-constant face gradients without any metric.
corner_gradients = (time & face_sides) / (2 * prisms)                        # [corners] Plane+[faces]
gradient = (spdiag(corner_gradients * Scalar) * mesh.incidence[2, 0]).sum(axis=0)  # (Plane+ <- Scalar+)[faces <- vertices]

# The exterior derivative of each edge's Whitney 1-form is twice the wedge of its endpoints'
# gradients, matching the constant Whitney 2-form of unit integral over the face patch.
first, second = corner_gradients[[1, 2, 0]], corner_gradients[[2, 0, 1]]     # [corners] Plane+[faces], [corners] Plane+[faces]
curl = (spdiag(2 * (first ^ second) * Scalar) * mesh.boundary[2, 1]).sum(axis=0)  # (Magnetic+ <- Scalar+)[faces <- edges]

# The vacuum takes each face's field to the excitation it causes, weighted by the face's area; pulled
# back through the curl, it gives the curl stiffness, from edge readings to the edge currents.
vacuum = spdiag(areas * Magnetic.dual())                                     # (Patch- <- Magnetic+)[faces <- faces]
curl_stiffness = curl.adjugate()(vacuum(curl))                               # (Amount- <- Scalar+)[edges <- edges]

In [ ]:
abscissa, ordinate = mv.x & vertices, mv.y & vertices                        # Scalar+[vertices], Scalar+[vertices]
radial_envelope = (1 - vertices.squared()).squared()                         # Scalar+[vertices]
quadrupole = 1 + ellipticity * (abscissa.squared() - ordinate.squared())     # Scalar+[vertices]
stream = swirl_rate * radial_envelope * quadrupole                           # Scalar+[vertices]
face_velocity = gradient(stream) ^ normals                                   # Point-[faces]

# Pairing the unit bivector with the displacement from each lobe centre gives a circulating 1-form.
displacements = edge_midpoints.batch()[:, None] - lobe_centres[None, :]      # [edges, lobes] Point-
envelopes = (-displacements.squared() / (2 * lobe_radius**2)).exp()          # [edges, lobes] Scalar+
potential = ((envelopes * lobe_strengths) * (mv.xy & displacements)).sum(axis=-1).field()  # Plane+[edges]
start_cochain = potential & edges                                            # Scalar+[edges]

## Flux winding and diffusion

Neighbouring triangles agree on the area swept across their shared edge, so the field leaving one enters the other. The ideal evolution preserves the total flux and field energy; resistivity dissipates energy while preserving total flux.


In [ ]:
resistivities = np.array([0.0, 0.015])                                       # [cases]
dt = 0.0027
n_steps = 356
n_steps_per_frame = 4
frame_duration_ms = 40

In [ ]:
# Averaging adjacent face fields onto each edge preserves exactness of uniform 2-forms and
# makes the ideal Lie advection rate skew-symmetric in the 2-form magnetic energy.
face_to_edge = ~mesh.incidence[2, 1].sum(axis=0)                             # Scalar+[edges <- faces]
edge_counts = face_to_edge * as_scalar(np.ones(len(mesh.faces))).field()     # Scalar+[edges]
to_edges = spdiag(1 / edge_counts) * face_to_edge                            # Scalar+[edges <- faces]
edge_velocity_map = to_edges * Point                                         # (Point- <- Point-)[edges <- faces]
edge_curl = (to_edges * Magnetic)(curl)                                      # (Magnetic+ <- Scalar+)[edges <- edges]

edge_velocity = edge_velocity_map(face_velocity)                             # Point-[edges]
# The world sheet each fixed edge sweeps through time, and the one it sweeps carried by the flow.
fixed_sheets = time & edges                                                  # Patch-[edges]
comoving_sheets = (time + edge_velocity) & edges                             # Patch-[edges]

# A fixed edge's reading changes by the field on its world sheet; ideal Ohm's law zeroes the field on the
# comoving sheet, leaving the patch of space between the two, which the magnetic plane alone reads.
mesh_swept_patches = fixed_sheets - comoving_sheets                          # Patch-[edges]
# The prescribed velocity makes transport a fixed map from edge readings to their rates.
advection = spdiag(Magnetic & mesh_swept_patches)(edge_curl)                 # (Scalar+ <- Scalar+)[edges <- edges]

# A unit resistance on each interior edge turns its current into the resistive electric field, a plane
# through time, read on the edge's world sheet: a rate of its reading. Restricting the response to
# interior edges keeps boundary circulation fixed under resistivity.
interior = ~SparseExtensor.selection(mv.context, mesh.interior_edges, len(mesh.edges)) * Scalar  # (Scalar+ <- Scalar+)[edges <- interior]
unit_resistance = spdiag(as_scalar(np.ones(len(mesh.interior_edges))).field() * Amount.dual_inverse())  # (Scalar+ <- Amount-)[interior <- interior]
curl_laplacian = interior(unit_resistance(interior.adjugate()(curl_stiffness)))  # (Scalar+ <- Scalar+)[edges <- edges]

rate = advection - curl_laplacian * resistivities[:, None]                   # [cases] (Scalar+ <- Scalar+)[edges <- edges]

In [ ]:
def advect(initial: Scalar, rate: Evolution, dt: float, n_steps: int) -> Iterator[Scalar]:
    """Successive edge 1-form cochains along fourth-order Runge-Kutta steps."""
    cochain = initial                                                        # [cases] Scalar+[edges]
    yield cochain
    for _ in range(n_steps):
        first_rate = rate(cochain)                                           # [cases] Scalar+[edges]
        second_rate = rate(cochain + (dt / 2) * first_rate)                  # [cases] Scalar+[edges]
        third_rate = rate(cochain + (dt / 2) * second_rate)                  # [cases] Scalar+[edges]
        fourth_rate = rate(cochain + dt * third_rate)                        # [cases] Scalar+[edges]
        cochain = cochain + (dt / 6) * (first_rate + 2 * second_rate + 2 * third_rate + fourth_rate)  # [cases] Scalar+[edges]
        yield cochain


batched_start = stack((start_cochain, start_cochain))                        # [cases] Scalar+[edges]
# Retain scalar diagnostics and the final state; mesh fields stream through one at a time.
energy_trace, flux_trace = [], []
for cochain in advect(batched_start, rate, dt, n_steps):
    energy_trace.append((cochain & curl_stiffness(cochain)).sites.sum() / 2)
    flux_trace.append(d1(cochain).sites.sum())
energies = stack(energy_trace)                                               # [steps + 1, cases] Scalar+
total_fluxes = stack(flux_trace)                                             # [steps + 1, cases] Scalar+
final_flux_density = flux_density(cochain)                                   # [cases] Scalar+[faces]
times = np.arange(n_steps + 1) * dt

save_figure(
    render.draw_advection_comparison(mesh, vertices, final_flux_density,
                                     times, energies, total_fluxes),
    "lie_advection_snapshots",
);

The ideal conductor on the left winds the lobes into thin layers, which diffusion smooths in the middle disk; orange and blue mark positive and negative flux density, and contours join equal values of it. On the right, relative to their initial values, the ideal energy (dark) holds, the resistive energy (orange) falls, and the resistive total flux (dashed teal) holds.

In the matrix notation of finite element exterior calculus, the curl-curl stiffness reads $S_1 = d_1^\top M_2 d_1$, with $M_2 = \operatorname{diag}(1 / |f|)$. Heumann and Hiptmair's [*Extrusion Contraction Upwind Schemes for Convection-Diffusion Problems on Differential Forms*](https://sam.math.ethz.ch/sam_reports/reports_final/reports2011/2011-17.pdf) develops the related swept-cell interpretation of transport.


In [ ]:
print("Relative final magnetic energy (ideal, resistive):", (energies[-1] / energies[0]).to_array())

## Evolving flux layers

Differential rotation stretches the lobes around the core. The ideal conductor on the left retains the thin layers; diffusion in the conductor on the right broadens them and reduces their intensity. The fixed colour scale makes that loss visible. This cell takes about ten seconds, most of it drawing the animation.


In [ ]:
sampled_states = islice(advect(batched_start, rate, dt, n_steps), 0, None, n_steps_per_frame)
frames = (flux_density(state) for state in sampled_states)
show_animation(
    render.animate_advection(mesh, vertices, frames),
    "lie_advection_flow",
    frame_duration_ms,
)


In [ ]:
# checks
# Associativity of the regressive product equates the world-sheet flux to the interior-product line integral.
interior_line_integrals = (face_bivector & edge_motions) & sides
np.testing.assert_allclose((swept_fluxes - interior_line_integrals).kernel, 0.0, atol=1e-13)
np.testing.assert_allclose(endpoint_differences.sum(axis=0).kernel, 0.0, atol=1e-15)

# Cartan's decomposition matches the exact material derivative of the edge line integrals.
plus_corners = corners + 0.5 * corner_motions
minus_corners = corners - 0.5 * corner_motions
plus_sides = plus_corners[[2, 0, 1]] - plus_corners[[1, 2, 0]]
minus_sides = minus_corners[[2, 0, 1]] - minus_corners[[1, 2, 0]]
plus_midpoints = (plus_corners[[1, 2, 0]] + plus_corners[[2, 0, 1]]) / 2
minus_midpoints = (minus_corners[[1, 2, 0]] + minus_corners[[2, 0, 1]]) / 2
plus_hats = 1 + (gradients[:, None] & (plus_midpoints[None, :] - corners[:, None]))
minus_hats = 1 + (gradients[:, None] & (minus_midpoints[None, :] - corners[:, None]))
plus_forms = (plus_hats * corner_forms[:, None]).sum(axis=0)
minus_forms = (minus_hats * corner_forms[:, None]).sum(axis=0)
material_rates = (plus_forms & plus_sides) - (minus_forms & minus_sides)
np.testing.assert_allclose(((endpoint_differences + swept_fluxes) - material_rates).kernel, 0.0, atol=1e-13)

# Across every mesh edge, the swept patch area equals minus the streamfunction difference.
d0 = mesh.d[0] * Scalar
np.testing.assert_allclose(((mv.xy & mesh_swept_patches) + d0(stream)).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((gradient(stream) & face_velocity).kernel, 0.0, atol=1e-12)

# Ideal advection has zero instantaneous magnetic energy production and conserves energy and flux over RK4.
ideal_initial_rate = advection(start_cochain)
np.testing.assert_allclose(
    (ideal_initial_rate & curl_stiffness(start_cochain)).sites.sum().kernel,
    0.0, atol=1e-12,
)
np.testing.assert_allclose((total_fluxes[:, 0] - total_fluxes[0, 0]).kernel, 0.0, atol=1e-13)
np.testing.assert_allclose((total_fluxes[:, 1] - total_fluxes[0, 1]).kernel, 0.0, atol=1e-14)
np.testing.assert_allclose((energies[:, 0] / energies[0, 0]).kernel, 1.0, rtol=1e-4)
assert np.all(np.diff(energies[:, 1].to_array()) < 0)